# 리포트 57 — 세 밴드에서 값이 다른 항은 λ² 와 σ 둘뿐이다

> ### 한 일
> **d = 1 km 한 점에서 세 조명원의 출력 SNR 을 항별로 분해하고, 밴드 쌍의 격차를 그 항으로 쪼갰다.**

### 결과
1. WiFi−LTE 격차 +3.54 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_total⟩ 는 λ² 항 -9.03 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_lambda2⟩ 와 σ 항 +12.57 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_sigma⟩ 의 합이다.
2. 공통항(EIRP·수신이득·확산·1/N₀·CPI·듀티·손실)은 세 밴드가 같은 값 +78.45 dB ⟨outputs/report05_derived.json : gap_1km.by_mode.W1.common⟩ 를 쓴다 — 밴드 쌍의 공통항 차는 +0.00 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_common⟩ 다.
3. 5기체 × 3쌍 15칸 중 σ 항이 λ² 항보다 큰 칸은 9 ⟨outputs/sigma_sensitivity.json : gap_decomposition.n_pairs_sigma_dominates⟩칸이고, σ-무관 축의 격차는 λ² 스프레드로 고정이다.
4. 헤드라인 칸을 구속하는 벽은 직접파 잔차이고(dpi_residual ⟨outputs/report13_freespace.json : solve.W1.limit⟩), 레이더 방정식 항등식 검사는 코드 경로와 dB 산술의 차이를 2.8e-14 dB ⟨outputs/verify_linkbudget.json : A_radar_equation.rows → |d_echo_dbarith_db| 최대⟩ 로 잡는다.
5. 기준신호가 CPI 를 다 채운다는 규약을 풀면 듀티 항이 살아난다 — 5G 는 LTE 대비 16.02 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.pair_gaps_db.L1-G1⟩ 를 더 치른다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 동작점 | d = 1 km · Mavic 4 Pro · 수신소자 1개. 점유 규약은 equal_psd ⟨outputs/report13_freespace.json : meta.link_budget.power_normalization.canonical_occupancy⟩ 다 |
| σ 항 | 기울기 앵커의 밴드별 Δσ 를 우리 PO 출력 위에 더한 값 — 앵커의 범위는 [편 24 «앵커가 받는 축»](24_anchor-mode.ipynb) 이 든다 |
| 항등식 검사 | 파형 3종 × 기체 2종에서 코드 경로와 dB 산술을 맞대 본다 — `benchmark/verify_linkbudget.py` |
| 듀티 축 | R90 경로에서 호출되지 않는 항이라 크기만 따로 싣는다 — 그 항을 켠 설정의 순위는 [편 61 «순위 강건성»](61_rank-durability.ipynb) 이 든다 |

### 재현

```bash
for D in mini5pro mavic4pro matrice4e phantom4 s1000plus; do PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_range.py --stage all --mode W1,L1,G1 --drone $D; done
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_linkbudget.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/sigma_sensitivity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/sigma_anchor.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/report13_freespace.json`, `outputs/verify_linkbudget.json`, `outputs/sigma_sensitivity.json`, `outputs/sigma_anchor.json`, `outputs/report05_derived.json` |
| 소요 | 링크버짓 선언 예산 5100 s ⟨outputs/report05_derived.json : runtime.declared_s⟩ · σ 민감도 15.2 s ⟨outputs/report05_derived.json : runtime.sigma_sensitivity_s⟩ |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 24 «앵커가 받는 축»](24_anchor-mode.ipynb) | A(f) 기울기는 측정에서, 레벨과 자세 패턴은 우리 계산에서 온다 |
| [편 46 «조명원 dB 원장»](46_cost-ledger.ipynb) | 점유 · λ² · 듀티 · PRF 의 항별 크기 |

---

## 감도사슬 — 밴드 격차를 항으로 분해한다

d = 1 km · Mavic 4 Pro · 수신소자 1개. 점유 규약은 equal_psd ⟨outputs/report13_freespace.json : meta.link_budget.power_normalization.canonical_occupancy⟩(자원요소 RE 하나당 같은 전력)이고, σ 항은 기울기 앵커의 밴드별 Δσ 를 더한 값이다. **세 밴드에서 값이 다른 항은 λ² 와 σ 둘뿐이다** — 나머지는 세 밴드가 같은 값을 쓴다.

| 항 | WiFi | LTE | 5G |
|---|---|---|---|
| 공통항 합 (EIRP·수신이득·확산·1/N₀·CPI·듀티·손실) | +78.45 | +78.45 | +78.45 |
| $\lambda^2$ | -24.80 | -15.77 | -21.34 |
| $\sigma$ (기울기 앵커, 공칭 헤딩) | -11.33 | -23.90 | -12.75 |
| **출력 SNR** | +42.32 | +38.78 | +44.36 |

출처 ⟨outputs/report13_freespace.json : ranges.mavic4pro.*.equal_psd.full_waveform_capture.by_N.1.budget_terms_db⟩
출처 ⟨outputs/sigma_anchor.json : drones.*.modes.slope_only.delta_db⟩

밴드 쌍의 격차를 그 두 항으로 쪼갠다. λ² 는 정의로 정확하고, σ 는 자세 로브 구조(방위를 돌릴 때 σ 가 솟는 봉우리와 꺼지는 골)가 만든다.

| 쌍 | 출력 SNR 차 | λ² 항 | σ 항 |
|---|---|---|---|
| W1-L1 | +3.54 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_total⟩ | -9.03 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_lambda2⟩ | +12.57 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-L1.d_sigma⟩ |
| W1-G1 | -2.04 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-G1.d_total⟩ | -3.46 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-G1.d_lambda2⟩ | +1.41 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.W1-G1.d_sigma⟩ |
| L1-G1 | -5.59 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.L1-G1.d_total⟩ | +5.57 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.L1-G1.d_lambda2⟩ | -11.16 dB ⟨outputs/report05_derived.json : gap_1km.by_pair.L1-G1.d_sigma⟩ |

5기체 × 3쌍 15칸에서 σ 항이 더 큰 칸은 9 ⟨outputs/sigma_sensitivity.json : gap_decomposition.n_pairs_sigma_dominates⟩칸이고, σ-무관 축의 격차는 λ² 스프레드 -9.03 ⟨outputs/sigma_sensitivity.json : gap_decomposition.axes_pair_gaps_db.W1-L1⟩ / -3.46 ⟨outputs/sigma_sensitivity.json : gap_decomposition.axes_pair_gaps_db.W1-G1⟩ / +5.57 dB ⟨outputs/sigma_sensitivity.json : gap_decomposition.axes_pair_gaps_db.L1-G1⟩ 로 고정이다.

![report05_pf1_gap](../outputs/figures/report05_pf1_gap.png)

**그림 1.** 밴드 격차를 만드는 항은 λ² 와 σ 중 무엇인가?

## 어느 벽이 거리를 정하나

헤드라인 칸을 구속하는 것은 직접파 잔차다(dpi_residual ⟨outputs/report13_freespace.json : solve.W1.limit⟩). ECA 억압 깊이를 바꾸면 거리가 이렇게 움직인다.

| ECA 깊이 | 40 dB | 60 dB | 90 dB | 완전 억압 |
|---|---|---|---|---|
| R90 | 2786 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.40.R_m⟩ | 7618 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.60.R_m⟩ | 9720 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.90.R_m⟩ | 9724 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.inf.R_m⟩ |

레이더 방정식 항등식 검사는 파형 3종 × 기체 2종 6 행 ⟨outputs/verify_linkbudget.json : A_radar_equation.rows → 길이⟩ 에서 코드 경로와 dB 산술의 차이를 2.8e-14 dB ⟨outputs/verify_linkbudget.json : A_radar_equation.rows → |d_echo_dbarith_db| 최대⟩ 로 잡는다.

## 듀티 축의 크기

위 사슬은 기준신호가 CPI 전체를 채운다는 규약에서 풀린다. 실제 점유가 만드는 듀티 항은 밴드마다 다르고, 그 크기를 여기 적는다 — 이 항을 켠 설정의 순위는 [편 61 «순위 강건성»](61_rank-durability.ipynb) 이 든다.

| 모드 | 기준신호 길이 T_ref | 프레임 M | 듀티 항 |
|---|---|---|---|
| WiFi | 5.20e-05 s ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.W1.T_ref_s⟩ | 100 ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.W1.M⟩ | -12.84 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.W1.duty_db⟩ |
| LTE | 1.00e-03 s ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.L1.T_ref_s⟩ | 100 ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.L1.M⟩ | +0.00 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.L1.duty_db⟩ |
| 5G | 5.00e-04 s ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.G1.T_ref_s⟩ | 5 ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.G1.M⟩ | -16.02 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.by_mode.G1.duty_db⟩ |

이 항을 넣으면 5G 는 LTE 대비 16.02 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.pair_gaps_db.L1-G1⟩ 를 더 치른다. 같은 값이 집안의 다른 산출물에도 있다 — WiFi 의 -12.84 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.duty_db.W1⟩ 는 `outputs/report4_fixups.json : packet_duty_db` 와 일치한다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 듀티 항을 R90 경로에 켜고 세 밴드를 다시 푼다 | 위 표의 -16.02 dB ⟨outputs/sigma_sensitivity.json : unapplied_duty_axis.duty_db.G1⟩ 가 순위에 주는 영향이 확정되고, 거리표의 듀티 행이 실측값이 된다 | `src/freespace_link.py` 의 duty_db_from_cpi → [편 60 «R90 표»](60_r90.ipynb) |